# Searching gRNA sites for SNPs in *An funestus*

---

> **Please try not to edit this copy of the notebook** - make a copy for yourself by going to `File / Save a Copy in Drive`

---

This notebook will plot a graph showing the location of SNPs and their proportion within your chosen samples from the malariagen database.

You will need:

* The AFUN number of your gene of interest

* The chromosome coordinates of your gRNA(s) in the AfunGA1 genome

If you have not used malariagen before, you will need to register for access [by filling in this form](https://forms.gle/kCqistorZyxaU4LP7). You will also need a Google account to access the data.

## Setting up the environment

You'll need to run this from the start every time you open the notebook.

In [ ]:
# Install malariagen
%pip install -q --no-warn-conflicts malariagen_data

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# Import packages
import malariagen_data
import pandas as pd
import plotly.io as pio
pio.renderers.default = "notebook+colab"
import allel
import numpy as np
from bokeh.plotting import figure, show, output_notebook
from bokeh.io import output_notebook
from bokeh.palettes import Spectral4
from bokeh.models import (
    ColumnDataSource, HoverTool, NumeralTickFormatter,
    WheelZoomTool, BoxAnnotation, Span, CustomJS, FixedTicker
)

In [ ]:
# Call the af1 dataset
af1 = malariagen_data.Af1(
    results_cache="~/malariagen_data_results_cache",
)
af1

<MalariaGEN Af1 API client>
Storage URL                           : gs://vo_afun_release_master_us_central1
Data releases available               : 1.0, 1.1, 1.2, 1.3, 1.4
Results cache                         : /root/malariagen_data_results_cache
Cohorts analysis                      : 20240515
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.11.0
Client location                       : Iowa, United States (Google Cloud us-central1)
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 1.0, 1.1, 1.2, 1.3, 1.4
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.11.0/Af1.html

Note: check your malariagen_data version is based in the US - if it's not you'll need to start again as it won't work (Runtime > Disconnect and delete runtime)

In [ ]:
# Mount your Google Drive
try:
    from google.colab import drive
    drive.mount('drive')
except ImportError:
    pass

Mounted at drive


In [ ]:
# To determine which samples you want to look at, first lets call them all:
df_samples = af1.sample_metadata()
df_samples

,sample_id,partner_sample_id,contributor,country,location,year,month,latitude,longitude,sex_call,...,admin1_name,admin1_iso,admin2_name,taxon,cohort_admin1_year,cohort_admin1_month,cohort_admin1_quarter,cohort_admin2_year,cohort_admin2_month,cohort_admin2_quarter
0,VBS10123-4954STDY7089651,UG4A2016A2_88,Mara Lawniczak,Uganda,Busia,2016,6,0.466,34.089,F,...,Eastern Region,UG-E,Busia,funestus,UG-E_fune_2016,UG-E_fune_2016_06,UG-E_fune_2016_Q2,UG-E_Busia_fune_2016,UG-E_Busia_fune_2016_06,UG-E_Busia_fune_2016_Q2
1,VBS10132-4954STDY7089660,UG4A2016B4_66,Mara Lawniczak,Uganda,Busia,2016,6,0.466,34.089,F,...,Eastern Region,UG-E,Busia,funestus,UG-E_fune_2016,UG-E_fune_2016_06,UG-E_fune_2016_Q2,UG-E_Busia_fune_2016,UG-E_Busia_fune_2016_06,UG-E_Busia_fune_2016_Q2
2,VBS10154-4954STDY7089682,GS5,Mara Lawniczak,Kenya,Nyawiya,2017,4,-0.466,34.319,F,...,Homa Bay,KE-08,Suba North,funestus,KE-08_fune_2017,KE-08_fune_2017_04,KE-08_fune_2017_Q2,KE-08_Suba-North_fune_2017,KE-08_Suba-North_fune_2017_04,KE-08_Suba-North_fune_2017_Q2
3,VBS10156-4954STDY7089684,UG10201710A1_1,Mara Lawniczak,Uganda,Busia,2017,3,0.466,34.089,F,...,Eastern Region,UG-E,Busia,funestus,UG-E_fune_2017,UG-E_fune_2017_03,UG-E_fune_2017_Q1,UG-E_Busia_fune_2017,UG-E_Busia_fune_2017_03,UG-E_Busia_fune_2017_Q1
4,VBS10195-4954STDY7089723,UG17201717H2_16,Mara Lawniczak,Uganda,Mayuge,2017,6,0.458,33.481,F,...,Eastern Region,UG-E,Mayuge,funestus,UG-E_fune_2017,UG-E_fune_2017_06,UG-E_fune_2017_Q2,UG-E_Mayuge_fune_2017,UG-E_Mayuge_fune_2017_06,UG-E_Mayuge_fune_2017_Q2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3959,SAMN27766214,Dirze_02-06897_Kiribina,Scott Small,Burkina Faso,Dirze,2002,12,11.400,-0.600,F,...,Centre-Sud,BF-07,Zoundweogo,funestus,BF-07_fune_2002,BF-07_fune_2002_12,BF-07_fune_2002_Q4,BF-07_Zoundweogo_fune_2002,BF-07_Zoundweogo_fune_2002_12,BF-07_Zoundweogo_fune_2002_Q4
3960,SAMN27766216,Dirze_02-07231_Kiribina,Scott Small,Burkina Faso,Dirze,2002,12,11.400,-0.600,F,...,Centre-Sud,BF-07,Zoundweogo,funestus,BF-07_fune_2002,BF-07_fune_2002_12,BF-07_fune_2002_Q4,BF-07_Zoundweogo_fune_2002,BF-07_Zoundweogo_fune_2002_12,BF-07_Zoundweogo_fune_2002_Q4
3961,SAMN27766217,Kuiti_00-01518_Kiribina,Scott Small,Burkina Faso,Kuiti,2000,9,12.280,-1.410,F,...,Centre,BF-03,Kadiogo,funestus,BF-03_fune_2000,BF-03_fune_2000_09,BF-03_fune_2000_Q3,BF-03_Kadiogo_fune_2000,BF-03_Kadiogo_fune_2000_09,BF-03_Kadiogo_fune_2000_Q3
3962,SAMN27766218,Pehele_02-06339_Kiribina,Scott Small,Burkina Faso,Pehele,2002,11,12.250,-1.180,F,...,Centre,BF-03,Kadiogo,funestus,BF-03_fune_2002,BF-03_fune_2002_11,BF-03_fune_2002_Q4,BF-03_Kadiogo_fune_2002,BF-03_Kadiogo_fune_2002_11,BF-03_Kadiogo_fune_2002_Q4


## Specifying samples of interest

Looking for SNPs in your target region in the entire malariagen database would take a long time and a lot of computing power - therefore we need to narrow down the search to the most relevant samples.

You can do this by examining the metadata and deciding which samples are most important - for example, which taxa, which country or countries, or samples collected within a certain time period. You can use any columns in the metadata table to filter the samples - but these are the main three you'll probably need:

* `country`
* `year`
* `taxon` (main options: `gambiae`, `arabiensis`, `coluzzi`, plus some rarer/more niche strains)


To investigate the different possible options within a metadata column, try using the `.groupby()` function as shown below:

*note: all code in Python is case sensitive*

In [ ]:
# To show the number of samples in each country:
df_samples.groupby("sample_set").size()

,0
sample_set,
1178-VO-UG-LAWNICZAK-VMF00025,7
1188-VO-NIANG-NIEL-SN-2304-VMF00259,71
1190-VO-GH-AMENGA-ETEGO-VMF00013,1
1190-VO-GH-AMENGA-ETEGO-VMF00047,1
1191-VO-MULTI-OLOUGHLIN-VMF00106,1
1229-VO-GH-DADZIE-VMF00095,36
1230-VO-GA-CF-AYALA-VMF00045,50
1230-VO-MULTI-AYALA-AYDI-GA-2104,50
1231-VO-MULTI-WONDJI-VMF00043,320


In [ ]:
# To see the number of each taxon in one country (e.g. Burkina Faso)
df_samples.query("country == 'Burkina Faso'").groupby("taxon").size()

,0
taxon,
funestus,177


In [ ]:
# To see the number of each taxon in one country and also collected after 2014
df_samples.query("country == 'Burkina Faso' and year >= 2000").groupby("taxon").size()

,0
taxon,
funestus,177


Once you've figured out what samples you want to use, put them into the `query_parts` code below (making sure to keep any question marks, brackets etc in place in the code).

Modify the lines you want and put a `#` in front of the lines you don't use to make sure they're not included.

There are options for if you have a single group or multiple groups per metadata column you want to use.

In [ ]:
# These are the filters for your data - e.g. here we're selecting all funestus
# in Burkina Faso collected after 2000
# Add a # to the start of the lines you're not using
query_parts = [
    "taxon == 'funestus'",                      # Filter by taxon
    # "taxon in ['funestus', 'vaneedeni']",     # Or by multiple taxa
    "country == 'Burkina Faso'",                # Filter by single country
    # "country in ['Burkina Faso', 'Ghana']",   # Or by multiple countries
    "year >= 2000",                             # Filter by year (here since 2014)
]

In [ ]:
# Don't edit this cell - this filters the data down to only include sample sets
# with your query
query_str = " and ".join(query_parts)
df_filtered = df_samples.query(query_str)
sample_sets = df_filtered.groupby("sample_set").size().index.tolist()

# This is the list of sample sets being used downstream, based on your parameters,
# and the number of samples in each set
df_filtered.groupby("sample_set").size()

,0
sample_set,
small-2023,177


# Finding your target transcript

The SNP table will be generated based on a single transcript of your chosen gene. Finding the transcript ID within the *An. funestus* database is more difficult than with the *An. gambiae* genome, as there isn't standardisation of the transcripts with their respective genes in the AfunGA1 genome (like the AGAP naming system).

Therefore, to find the transcript ID, the best way is to find the chromosomal location of your gene using VectorBase. Note the chromosomes in AfunGA1 are arranged with left and right combined (2RL, 3RL and X).


In [ ]:
# Search the genome features database using coordinates
# (note the format: no commas in the numbers, etc.)
af1.genome_features(region="3RL:9759194-9768322").query("type == 'mRNA'")

,contig,source,type,start,end,score,strand,phase,ID,Parent,Note,description
1,3RL,VEuPathDB,mRNA,9759194,9768322,NaN,+,NaN,LOC125771265_t1,LOC125771265,NaN,"serine protease nudel, transcript variant X1"
4,3RL,VEuPathDB,mRNA,9759195,9768322,NaN,+,NaN,LOC125771265_t2,LOC125771265,NaN,"serine protease nudel, transcript variant X2"


In [ ]:
# Change this to the transcript ID from the table above (you may need to do some
# research to determine which transcript to use - or test them all)
transcript = "LOC125771265_t1"

# Running the code for your chosen sample sets and gene

In [ ]:
# Run this without editing to produce a table of all SNPs in your transcript
snp_allele_freqs_df = af1.snp_allele_frequencies(
    transcript=transcript,
    cohorts="admin1_year",
    sample_sets=sample_sets,
    drop_invariant=False,
)

# Filter out any SNPs that are present at <5% in the chosen samples (edit this
# if you want)
snp_filt_df = snp_allele_freqs_df.query("max_af >= 0.05")

Load SNP genotypes:   0%|          | 0/9 [00:00<?, ?it/s]

Compute allele frequencies:   0%|          | 0/6 [00:00<?, ?it/s]

Compute SNP effects:   0%|          | 0/27387 [00:00<?, ?it/s]

## Plotting function - DON'T CHANGE, JUST RUN

In [ ]:
# This cell contains the plotting functions - no need to edit it, just run it
import warnings
from bokeh.io import output_notebook, show
from bokeh.layouts import gridplot, column
from bokeh.models import (ColumnDataSource, HoverTool, WheelZoomTool, BoxAnnotation,
                          Span, NumeralTickFormatter, CustomJS, Range1d, Slider, Div)
from bokeh.plotting import figure

BASES = ['A', 'C', 'G', 'T']

# ---------- shared helpers ----------
def _get_api(api):
    """Use the malariagen API passed in, or find ag3 / af1 in the notebook."""
    if api is not None:
        return api
    for name in ("ag3", "af1"):
        if name in globals():
            return globals()[name]
    raise NameError("No malariagen API found - pass api=ag3 (or api=af1)")

def _get_transcript(transcript):
    if transcript is not None:
        return transcript
    if "transcript" in globals():
        return globals()["transcript"]
    raise NameError("No transcript given - pass transcript='AGAP...-RA'")

def _per_position(snp_df):
    """One row per position; max_af = summed alt allele freq (capped at 1)."""
    d = snp_df.reset_index()
    out = (d.groupby('position', as_index=False)
             .agg(max_af=('max_af', 'sum'))
             .sort_values('position'))
    out['max_af'] = out['max_af'].clip(upper=1)
    return out

def _per_position_bases(snp_df, freq_col='max_af'):
    """One row per position, with a frequency column for each of A/C/G/T."""
    d = snp_df.reset_index()
    alt = (d.pivot_table(index='position', columns='alt_allele', values=freq_col,
                         aggfunc='sum', fill_value=0)
             .reindex(columns=BASES, fill_value=0))
    alt.columns.name = None
    ref = d.groupby('position')['ref_allele'].first().reindex(alt.index)

    alt_total = alt.sum(axis=1).clip(upper=1)
    out = alt.copy()
    for b in BASES:                             # reference base gets the remainder
        is_ref = ref == b
        out.loc[is_ref, b] = 1 - alt_total[is_ref]

    out = out.reset_index()
    out['ref_allele'] = ref.values
    out['alt_freq'] = alt_total.values
    out['ref_freq'] = 1 - out['alt_freq']
    out['max_alt'] = alt.max(axis=1).values     # most common alt allele, used by the slider
    return out

def _x_range_from_snps(positions, pad_frac=0.02, min_pad=10):
    lo, hi = int(positions.min()), int(positions.max())
    pad = max((hi - lo) * pad_frac, min_pad)
    start, end = lo - pad, hi + pad
    return Range1d(start=start, end=end, bounds=(start, end))

def _base_figure(title, y_label, y_range, x_range, height=250):
    wheel_zoom_x = WheelZoomTool(dimensions="width")
    p = figure(width=900, height=height, y_axis_label=y_label,
               tools="pan,box_zoom,reset,save", title=title,
               x_range=x_range, y_range=y_range)
    p.add_tools(wheel_zoom_x)
    p.toolbar.active_scroll = wheel_zoom_x
    p.xaxis.formatter = NumeralTickFormatter(format="0,0")
    p.xaxis.major_label_text_font_size = "10pt"
    p.xaxis.minor_tick_line_color = None
    return p

def _add_gRNA_highlights(p, gRNA_start_end):
    if not gRNA_start_end:
        return
    for start, end in gRNA_start_end:
        p.add_layout(BoxAnnotation(left=start-0.5, right=end+0.5, fill_alpha=0.2,
                                   fill_color="green", level="underlay",
                                   editable=False, propagate_hover=True))
        for loc in (start-0.5, end+0.5):
            p.add_layout(Span(location=loc, dimension='height', line_color='green',
                              line_dash='dashed', line_width=2, level='overlay'))

def _add_dynamic_ticks(x_range, axes):
    callback = CustomJS(args=dict(axes=axes), code="""
        const start = Math.floor(cb_obj.start);
        const end = Math.ceil(cb_obj.end);
        const range = end - start;
        let spacing;
        if (range > 100000) spacing = 10000;
        else if (range > 10000) spacing = 1000;
        else if (range > 1000) spacing = 100;
        else if (range > 100) spacing = 10;
        else spacing = 1;
        let ticks = [];
        for (let i = Math.ceil(start/spacing)*spacing; i <= end; i += spacing) ticks.push(i);
        for (const ax of axes) ax.ticker = ticks;
    """)
    x_range.js_on_change('start', callback)
    x_range.js_on_change('end', callback)

def _with_gene_track(p, api, transcript, gRNA_start_end, gene_height=120, widgets=None):
    fig_gene = api.plot_transcript(transcript, x_range=p.x_range, width=900,
                                   height=gene_height, show=False)
    fig_gene.x_range = p.x_range
    fig_gene.xaxis.axis_label = "Position"
    fig_gene.xaxis.formatter = NumeralTickFormatter(format="0,0")
    _add_gRNA_highlights(fig_gene, gRNA_start_end)
    _add_dynamic_ticks(p.x_range, [p.xaxis[0], fig_gene.xaxis[0]])

    # Both figures set their own default pan/scroll tools; Bokeh keeps one and warns.
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="found multiple competing values")
        grid = gridplot([p, fig_gene], ncols=1, toolbar_location="above", merge_tools=True)

    output_notebook(hide_banner=True)   # needed in the same cell as show() in Colab
    show(column(*widgets, grid) if widgets else grid)


# ---------- 1. allele frequency bars + gene track, with threshold slider ----------
def plot_snps_gRNA(df, transcript=None, gRNA_start_end=None, min_af=0.05, slider_max=0.5,
                   freq_col='max_af', api=None, ref_color="#999999", alt_color="#ff3333"):
    """
    df: UNFILTERED output of snp_allele_frequencies (drop_invariant=False).
    transcript: defaults to the notebook's `transcript` variable.
    min_af: starting threshold for the slider. slider_max: top of the slider.
    freq_col: 'max_af' (max across cohorts) or a specific cohort, e.g. 'frq_<cohort>'.
    api: defaults to ag3 / af1, whichever exists in the notebook.
    """
    api, transcript = _get_api(api), _get_transcript(transcript)

    full_df = _per_position_bases(df, freq_col=freq_col)
    full_df['top'] = 1
    full_source = ColumnDataSource(full_df)
    shown_source = ColumnDataSource(full_df[full_df['max_alt'] >= min_af])

    positions = full_df['position'].values
    min_dist = int(min(positions[1:] - positions[:-1])) if len(positions) > 1 else 1
    bar_width = min_dist * 0.8

    p = _base_figure(f"Allele Frequencies by Position in {transcript}",
                     "Allele frequency", (0, 1), _x_range_from_snps(full_df['position']))

    p.vbar_stack(stackers=['ref_freq', 'alt_freq'], x='position', width=bar_width,
                 color=[ref_color, alt_color], legend_label=['Reference', 'Alternative'],
                 source=shown_source)

    hover_bar = p.vbar(x='position', top='top', width=bar_width, source=shown_source,
                       fill_alpha=0, line_alpha=0)
    p.add_tools(HoverTool(
        tooltips=[("Position", "@position{0,0}"),
                  ("Reference base", "@ref_allele"),
                  ("A", "@A{0.000}"),
                  ("C", "@C{0.000}"),
                  ("G", "@G{0.000}"),
                  ("T", "@T{0.000}"),
                  ("Reference freq", "@ref_freq{0.00}"),
                  ("Alternative freq", "@alt_freq{0.00}")],
        renderers=[hover_bar], mode='vline'))

    p.legend.location = "top_right"
    p.legend.click_policy = "hide"
    _add_gRNA_highlights(p, gRNA_start_end)

    n_start = int((full_df['max_alt'] >= min_af).sum())
    slider = Slider(start=0, end=slider_max, value=min_af, step=0.005, width=400,
                    format="0.000", title="Minimum alternative allele frequency")
    counter = Div(text=f"{n_start} positions shown")

    slider.js_on_change('value', CustomJS(
        args=dict(full=full_source, shown=shown_source, counter=counter), code="""
        const thr = cb_obj.value;
        const f = full.data;
        const keys = Object.keys(f);
        const out = {};
        for (const k of keys) out[k] = [];
        let n = 0;
        for (let i = 0; i < f.max_alt.length; i++) {
            if (f.max_alt[i] >= thr) {
                for (const k of keys) out[k].push(f[k][i]);
                n++;
            }
        }
        shown.data = out;
        counter.text = `${n} positions shown`;
    """))

    _with_gene_track(p, api, transcript, gRNA_start_end, widgets=[slider, counter])


# ---------- 2. conservation line + gene track, with window slider ----------
def plot_conservation_gRNA(df, transcript=None, gRNA_start_end=None, window=20,
                           window_max=200, api=None, line_color="#1f77b4"):
    """
    Conservation = 1 - summed alt allele freq at each position.
    df: UNFILTERED output of snp_allele_frequencies (drop_invariant=False).
    window: starting rolling-mean window in bp (adjustable with the slider);
            None = raw per-site values only, no slider.
    window_max: upper end of the window slider.
    """
    api, transcript = _get_api(api), _get_transcript(transcript)

    df = _per_position(df)
    df['conservation'] = 1 - df['max_af']
    if window:
        df['conservation_smooth'] = (df['conservation']
                                     .rolling(window, center=True, min_periods=1).mean())
    source = ColumnDataSource(df)

    p = _base_figure(f"Conservation across {transcript}",
                     "Conservation (ref allele freq)", (0, 1.02),
                     _x_range_from_snps(df['position']))

    raw = p.line('position', 'conservation', source=source, color=line_color,
                 line_width=1.5, alpha=0.4 if window else 1, legend_label="Per site")
    hover_renderers = [raw]
    tooltips = [("Position", "@position{0,0}"), ("Conservation", "@conservation{0.000}")]
    widgets = None

    if window:
        smooth = p.line('position', 'conservation_smooth', source=source,
                        color="#d62728", line_width=2.5, legend_label="Rolling mean")
        hover_renderers = [smooth]
        tooltips.append(("Rolling mean", "@conservation_smooth{0.000}"))

        slider = Slider(start=1, end=max(window_max, window), value=window, step=1,
                        width=400, title="Rolling mean window (bp)")
        # Centred rolling mean, matching pandas rolling(center=True, min_periods=1)
        slider.js_on_change('value', CustomJS(args=dict(source=source), code="""
            const w = cb_obj.value;
            const c = source.data['conservation'];
            const n = c.length;
            const pre = new Float64Array(n + 1);
            for (let i = 0; i < n; i++) pre[i + 1] = pre[i] + c[i];
            const half = Math.floor(w / 2);
            const sm = new Array(n);
            for (let i = 0; i < n; i++) {
                const lo = Math.max(0, i + half - w + 1);
                const hi = Math.min(n - 1, i + half);
                sm[i] = (pre[hi + 1] - pre[lo]) / (hi - lo + 1);
            }
            source.data['conservation_smooth'] = sm;
            source.change.emit();
        """))
        widgets = [slider]

    p.add_tools(HoverTool(tooltips=tooltips, renderers=hover_renderers, mode='vline'))
    p.legend.location = "bottom_right"
    p.legend.click_policy = "hide"
    _add_gRNA_highlights(p, gRNA_start_end)
    _with_gene_track(p, api, transcript, gRNA_start_end, widgets=widgets)

## Running the final plotting function

You'll need to know the start and end locations of your guide RNA in the chromosome your gene is located in

e.g. in this case the AGAP007280 gRNA is located at 2L:44,994,012-44,994,029

BUT but you need to format it as just the number, so `(44994012, 44994029)`



---



Note: you can put in multiple gRNA locations by listing them with commas within the square brackets - like this:

`gRNA_regions = [(44994012, 44994029), (44994112, 44994129)]`

In [ ]:
# Your gRNA regions
gRNA_regions = [(2358158, 2358177)]

### **Allele frequencies with a slider**

Edit `gRNA_start_end` to match your gRNA start and end position on the chromosome.

The Minimum alternative allele frequency slider can be used to filter out sites that have SNPs at different levels - e.g. the default is 0.05, so it will show all sites that have a SNP present in 5% or more of the population.

You can scroll to zoom into the graph, and hover over each individual bar to see the frequency of each base at each site. You can also see the gene plot underneath showing UTRs, CDS and intronic regions.

In [ ]:
plot_snps_gRNA(snp_allele_freqs_df, transcript, gRNA_start_end=gRNA_regions)

### **Conservation scores**

Again edit `gRNA_start_end` to point to your gRNA region defined earlier. This shows a similar thing to the above plot but also shows a rolling average of the frequency of alternate alleles at each site; this is set to calculate the average over 20 bp windows by default, but you can change this number by changing `window=20`.

In [ ]:
plot_conservation_gRNA(snp_allele_freqs_df, transcript, gRNA_start_end=gRNA_regions, window=20)

*2025.09.16 Poppy Pescod*

---

*Updates*

*2026.11.26 - added: visible bases to the bar mouse overs, the gene regions underneath, a scrollbar to filter for allele frequency, and a plot_conservation_score plot. Also updated the link to the access request.*